In [ ]:
###PARSING RELEVANT SKILLS USING JD + skills ranked with difficulty levels
import time as time

import pandas as pd

start_time=time.time()

r_df = pd.read_csv("hf://datasets/opensporks/resumes/Resume/Resume.csv") #resume dataset

print("\nresume dataset columns:", r_df.columns)


splits = {'train': 'train.csv', 'test': 'test.csv'}
rj_df = pd.read_csv("hf://datasets/cnamuangtoun/resume-job-description-fit/" + splits["train"]) #resume-job description dataset

print("\nresume-job description dataset columns:", rj_df.columns)


jss_df=pd.read_parquet("hf://datasets/batuhanmtl/job-skill-set/data/train-00000-of-00001.parquet") #job skill sets dataset
print("\njob skill set dataset columns:", jss_df.columns)


import pandas as pd
import kagglehub
import os


resume dataset columns: Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')

resume-job description dataset columns: Index(['resume_text', 'job_description_text', 'label'], dtype='str')

job skill set dataset columns: Index(['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'], dtype='str')


In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("batuhanmutlu/job-skill-set")

print("Dataset path:", path)
print("Files:", os.listdir(path))

import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("batuhanmutlu/job-skill-set")

# Load the CSV file
file_path = os.path.join(path, "all_job_post.csv")

kj_df = pd.read_csv(file_path)

print("\nkaggle job skill set dataset columns:", kj_df.columns)

Dataset path: /Users/era/.cache/kagglehub/datasets/batuhanmutlu/job-skill-set/versions/2
Files: ['all_job_post.csv']

kaggle job skill set dataset columns: Index(['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'], dtype='str')


In [ ]:
print("all datasets loaded successfully")

all datasets loaded successfully


In [ ]:
"""
Job Posting Skill Model
=======================
- Parse relevant skills from job descriptions
- Rank skills by relevance score + difficulty level
- Match job skills against resume skills
- Identify and prioritize missing skills
"""

'\nJob Posting Skill Model\n=======================\n- Parse relevant skills from job descriptions\n- Rank skills by relevance score + difficulty level\n- Match job skills against resume skills\n- Identify and prioritize missing skills\n'

In [ ]:
### JOB SKILL MODEL
import pandas as pd
import re
import json
from collections import Counter
from dataclasses import dataclass, field
from typing import Optional


# ─────────────────────────────────────────────
# 1. SKILL TAXONOMY  (extend as needed)
# ─────────────────────────────────────────────

# Difficulty tiers: 1 = Beginner  2 = Intermediate  3 = Advanced  4 = Expert
SKILL_TAXONOMY: dict[str, dict] = {
    # ── Programming Languages ──────────────────
    "python":           {"category": "Programming",      "difficulty": 2},
    "r":                {"category": "Programming",      "difficulty": 2},
    "sql":              {"category": "Data Engineering", "difficulty": 2},
    "java":             {"category": "Programming",      "difficulty": 3},
    "scala":            {"category": "Programming",      "difficulty": 3},
    "javascript":       {"category": "Programming",      "difficulty": 2},
    "typescript":       {"category": "Programming",      "difficulty": 2},
    "c++":              {"category": "Programming",      "difficulty": 3},
    "go":               {"category": "Programming",      "difficulty": 3},
    "rust":             {"category": "Programming",      "difficulty": 4},
    "bash":             {"category": "Programming",      "difficulty": 2},
    "shell":            {"category": "Programming",      "difficulty": 2},

    # ── ML / AI ────────────────────────────────
    "machine learning": {"category": "ML/AI",            "difficulty": 3},
    "deep learning":    {"category": "ML/AI",            "difficulty": 4},
    "nlp":              {"category": "ML/AI",            "difficulty": 4},
    "computer vision":  {"category": "ML/AI",            "difficulty": 4},
    "reinforcement learning": {"category": "ML/AI",      "difficulty": 4},
    "llm":              {"category": "ML/AI",            "difficulty": 4},
    "pytorch":          {"category": "ML/AI",            "difficulty": 3},
    "tensorflow":       {"category": "ML/AI",            "difficulty": 3},
    "keras":            {"category": "ML/AI",            "difficulty": 2},
    "scikit-learn":     {"category": "ML/AI",            "difficulty": 2},
    "xgboost":          {"category": "ML/AI",            "difficulty": 2},
    "hugging face":     {"category": "ML/AI",            "difficulty": 3},
    "transformers":     {"category": "ML/AI",            "difficulty": 4},

    # ── Data Engineering ───────────────────────
    "spark":            {"category": "Data Engineering", "difficulty": 3},
    "hadoop":           {"category": "Data Engineering", "difficulty": 3},
    "kafka":            {"category": "Data Engineering", "difficulty": 3},
    "airflow":          {"category": "Data Engineering", "difficulty": 3},
    "dbt":              {"category": "Data Engineering", "difficulty": 2},
    "etl":              {"category": "Data Engineering", "difficulty": 2},
    "data pipeline":    {"category": "Data Engineering", "difficulty": 3},
    "data warehouse":   {"category": "Data Engineering", "difficulty": 3},
    "snowflake":        {"category": "Data Engineering", "difficulty": 2},
    "redshift":         {"category": "Data Engineering", "difficulty": 2},
    "bigquery":         {"category": "Data Engineering", "difficulty": 2},
    "databricks":       {"category": "Data Engineering", "difficulty": 3},

    # ── Cloud ──────────────────────────────────
    "aws":              {"category": "Cloud",            "difficulty": 2},
    "azure":            {"category": "Cloud",            "difficulty": 2},
    "gcp":              {"category": "Cloud",            "difficulty": 2},
    "docker":           {"category": "Cloud/DevOps",     "difficulty": 2},
    "kubernetes":       {"category": "Cloud/DevOps",     "difficulty": 3},
    "terraform":        {"category": "Cloud/DevOps",     "difficulty": 3},
    "ci/cd":            {"category": "Cloud/DevOps",     "difficulty": 2},

    # ── Statistics / Analytics ─────────────────
    "statistics":       {"category": "Analytics",        "difficulty": 2},
    "a/b testing":      {"category": "Analytics",        "difficulty": 2},
    "hypothesis testing": {"category": "Analytics",      "difficulty": 2},
    "regression":       {"category": "Analytics",        "difficulty": 2},
    "time series":      {"category": "Analytics",        "difficulty": 3},
    "tableau":          {"category": "Analytics",        "difficulty": 1},
    "power bi":         {"category": "Analytics",        "difficulty": 1},
    "excel":            {"category": "Analytics",        "difficulty": 1},

    # ── Databases ──────────────────────────────
    "postgresql":       {"category": "Database",         "difficulty": 2},
    "mysql":            {"category": "Database",         "difficulty": 2},
    "mongodb":          {"category": "Database",         "difficulty": 2},
    "redis":            {"category": "Database",         "difficulty": 2},
    "elasticsearch":    {"category": "Database",         "difficulty": 3},
    "neo4j":            {"category": "Database",         "difficulty": 3},

    # ── Soft / Process ─────────────────────────
    "agile":            {"category": "Process",          "difficulty": 1},
    "scrum":            {"category": "Process",          "difficulty": 1},
    "git":              {"category": "Tools",             "difficulty": 1},
    "linux":            {"category": "Tools",             "difficulty": 2},
    "communication":    {"category": "Soft Skills",      "difficulty": 1},
    "leadership":       {"category": "Soft Skills",      "difficulty": 2},
    "problem solving":  {"category": "Soft Skills",      "difficulty": 2},
}

# Signals that boost relevance score of the NEXT noun/skill phrase
IMPORTANCE_SIGNALS: list[str] = [
    "required", "must have", "essential", "key skill", "strong",
    "proficient", "expert", "hands-on", "proven", "extensive",
    "minimum", "at least", "years of", "experience with",
    "knowledge of", "familiarity with", "background in",
]


# ─────────────────────────────────────────────
# 2. DATA CLASSES
# ─────────────────────────────────────────────

@dataclass
class RankedSkill:
    skill: str
    relevance_score: float        # 0-100
    difficulty_level: int         # 1-4
    difficulty_label: str         # Beginner / Intermediate / Advanced / Expert
    category: str
    in_resume: bool = False
    priority_rank: int = 0        # rank among MISSING skills (1 = fix first)


@dataclass
class SkillMatchReport:
    job_title: str
    matched_skills: list[RankedSkill] = field(default_factory=list)
    missing_skills: list[RankedSkill] = field(default_factory=list)
    match_score: float = 0.0      # weighted % of job skills present in resume

    def to_dict(self) -> dict:
        return {
            "job_title": self.job_title,
            "match_score": round(self.match_score, 1),
            "matched_skills": [vars(s) for s in self.matched_skills],
            "missing_skills": [vars(s) for s in self.missing_skills],
        }


# ─────────────────────────────────────────────
# 3. HELPERS
# ─────────────────────────────────────────────

DIFF_LABELS = {1: "Beginner", 2: "Intermediate", 3: "Advanced", 4: "Expert"}


def _normalize(text: str) -> str:
    return text.lower().strip()


def _extract_skills_from_text(text: str) -> dict[str, float]:
    """
    Scan text for known skills.
    Returns {skill_name: raw_boost} where boost reflects importance signals nearby.
    """
    text_lower = text.lower()
    found: dict[str, float] = {}

    # Build importance-signal positions
    signal_positions: list[int] = []
    for sig in IMPORTANCE_SIGNALS:
        for m in re.finditer(re.escape(sig), text_lower):
            signal_positions.append(m.start())

    for skill in SKILL_TAXONOMY:
        # Use word-boundary-aware search for multi-word and single-word skills
        pattern = r"\b" + re.escape(skill) + r"\b"
        matches = list(re.finditer(pattern, text_lower))
        if not matches:
            continue

        # Count mentions (frequency boost)
        freq = len(matches)

        # Check proximity to importance signals (within 120 chars)
        signal_boost = 0.0
        for m in matches:
            for sp in signal_positions:
                if abs(m.start() - sp) <= 120:
                    signal_boost += 1.0
                    break  # one boost per match

        raw_score = freq + signal_boost
        found[skill] = found.get(skill, 0) + raw_score

    return found


def _normalize_scores(raw: dict[str, float], max_score: float = 100.0) -> dict[str, float]:
    if not raw:
        return {}
    top = max(raw.values())
    if top == 0:
        return {k: 0.0 for k in raw}
    return {k: round((v / top) * max_score, 1) for k, v in raw.items()}


def _parse_resume_skills(resume_text: str) -> set[str]:
    """Return set of skill names found in the resume."""
    found = _extract_skills_from_text(resume_text)
    return set(found.keys())


# ─────────────────────────────────────────────
# 4. CORE MODEL
# ─────────────────────────────────────────────

def parse_job_skills(job_description: str, job_title: str = "Unknown") -> list[RankedSkill]:
    """
    Parse a job description and return skills ranked by relevance score.
    """
    raw = _extract_skills_from_text(job_description)
    normalized = _normalize_scores(raw)

    ranked: list[RankedSkill] = []
    for skill, score in sorted(normalized.items(), key=lambda x: -x[1]):
        meta = SKILL_TAXONOMY[skill]
        diff = meta["difficulty"]
        ranked.append(RankedSkill(
            skill=skill,
            relevance_score=score,
            difficulty_level=diff,
            difficulty_label=DIFF_LABELS[diff],
            category=meta["category"],
        ))
    return ranked


def match_skills(
    job_description: str,
    resume_text: str,
    job_title: str = "Unknown",
    top_n: int = 20,
) -> SkillMatchReport:
    """
    Full pipeline:
      1. Parse job skills with relevance scores
      2. Parse resume skills
      3. Compute match / gap
      4. Rank missing skills by (relevance × difficulty) — highest priority first
    """
    job_skills = parse_job_skills(job_description, job_title)[:top_n]
    resume_skills = _parse_resume_skills(resume_text)

    report = SkillMatchReport(job_title=job_title)
    total_weight = sum(s.relevance_score for s in job_skills) or 1.0
    matched_weight = 0.0

    for skill in job_skills:
        skill.in_resume = skill.skill in resume_skills
        if skill.in_resume:
            matched_weight += skill.relevance_score
            report.matched_skills.append(skill)
        else:
            report.missing_skills.append(skill)

    # Priority rank for missing skills:
    # higher relevance AND higher difficulty = learn sooner
    # Formula: priority_score = relevance_score * difficulty_level
    report.missing_skills.sort(
        key=lambda s: -(s.relevance_score * s.difficulty_level)
    )
    for rank, skill in enumerate(report.missing_skills, start=1):
        skill.priority_rank = rank

    report.match_score = (matched_weight / total_weight) * 100
    return report


# ─────────────────────────────────────────────
# 5. DISPLAY UTILITIES
# ─────────────────────────────────────────────

def print_report(report: SkillMatchReport) -> None:
    bar = "─" * 60
    print(f"\n{'═'*60}")
    print(f"  JOB: {report.job_title}")
    print(f"  MATCH SCORE: {report.match_score:.1f}%")
    print(f"{'═'*60}")

    print(f"\n✅  MATCHED SKILLS  ({len(report.matched_skills)})")
    print(bar)
    print(f"{'Skill':<28} {'Relevance':>10}  {'Difficulty':<14} {'Category'}")
    print(bar)
    for s in sorted(report.matched_skills, key=lambda x: -x.relevance_score):
        print(f"  {s.skill:<26} {s.relevance_score:>9.1f}  {s.difficulty_label:<14} {s.category}")

    print(f"\n❌  MISSING SKILLS  ({len(report.missing_skills)})  — ordered by learning priority")
    print(bar)
    print(f"{'#':<4} {'Skill':<28} {'Relevance':>10}  {'Difficulty':<14} {'Category'}")
    print(bar)
    for s in report.missing_skills:
        print(f"  {s.priority_rank:<3} {s.skill:<26} {s.relevance_score:>9.1f}  {s.difficulty_label:<14} {s.category}")

    print()


# ─────────────────────────────────────────────
# 6. BATCH ANALYSIS OVER DATASETS
# ─────────────────────────────────────────────

def analyze_dataset_sample(
    jss_df: pd.DataFrame,
    resume_text: str,
    n_jobs: int = 5,
    top_n_skills: int = 15,
) -> pd.DataFrame:
    """
    Run match_skills over n_jobs rows of the job-skill-set dataset.
    Returns a tidy DataFrame of results.
    """
    rows = []
    sample = jss_df.dropna(subset=["job_description", "job_title"]).head(n_jobs)

    for _, row in sample.iterrows():
        report = match_skills(
            job_description=row["job_description"],
            resume_text=resume_text,
            job_title=row.get("job_title", "Unknown"),
            top_n=top_n_skills,
        )
        for skill in report.matched_skills + report.missing_skills:
            rows.append({
                "job_title":        report.job_title,
                "job_category":     row.get("category", ""),
                "match_score":      report.match_score,
                "skill":            skill.skill,
                "relevance_score":  skill.relevance_score,
                "difficulty_level": skill.difficulty_level,
                "difficulty_label": skill.difficulty_label,
                "category":         skill.category,
                "in_resume":        skill.in_resume,
                "missing_priority": skill.priority_rank if not skill.in_resume else None,
            })

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────
# 7.  DEMO / QUICK TEST
# ─────────────────────────────────────────────

if __name__ == "__main__":

    SAMPLE_JD = """
    Senior Data Scientist – NLP & Machine Learning

    We are looking for a strong Python and SQL developer with extensive experience
    in machine learning and deep learning. The ideal candidate must have hands-on
    experience with PyTorch or TensorFlow and proven expertise in NLP, including
    transformer models and Hugging Face libraries.

    Required:
    - 5+ years of Python (expert level)
    - Strong background in statistics and hypothesis testing
    - Experience with Spark and distributed data pipelines
    - Familiarity with AWS or GCP cloud platforms
    - Knowledge of Docker and Kubernetes
    - Proficient in SQL and data warehouse solutions (Snowflake or Redshift)
    - Experience with Airflow or similar orchestration tools

    Nice to have:
    - Scala experience
    - Tableau or Power BI for stakeholder reporting
    - Agile/Scrum environment
    """

    SAMPLE_RESUME = """
    Skills: Python, SQL, scikit-learn, machine learning, statistics, hypothesis testing,
    Tableau, Git, Agile, PostgreSQL, Excel, AWS, Docker.
    Experience: 4 years building ML pipelines, ETL processes, A/B testing frameworks.
    """

    report = match_skills(
        job_description=SAMPLE_JD,
        resume_text=SAMPLE_RESUME,
        job_title="Senior Data Scientist – NLP & ML",
        top_n=20,
    )
    print_report(report)

    # Export to JSON
output_path = "skill_match_report.json"  # saves in your current working directory
with open(output_path, "w") as f:
    json.dump(report.to_dict(), f, indent=2)
print(f"JSON report saved → {output_path}")


════════════════════════════════════════════════════════════
  JOB: Senior Data Scientist – NLP & ML
  MATCH SCORE: 41.7%
════════════════════════════════════════════════════════════

✅  MATCHED SKILLS  (7)
────────────────────────────────────────────────────────────
Skill                         Relevance  Difficulty     Category
────────────────────────────────────────────────────────────
  python                         100.0  Intermediate   Programming
  sql                            100.0  Intermediate   Data Engineering
  machine learning               100.0  Advanced       ML/AI
  aws                             50.0  Intermediate   Cloud
  docker                          50.0  Intermediate   Cloud/DevOps
  statistics                      50.0  Intermediate   Analytics
  hypothesis testing              50.0  Intermediate   Analytics

❌  MISSING SKILLS  (13)  — ordered by learning priority
────────────────────────────────────────────────────────────
#    Skill                  

In [ ]:
### run model
# Cell 1 — run the model
report = match_skills(
    job_description=SAMPLE_JD,
    resume_text=SAMPLE_RESUME,
    job_title="Senior Data Scientist – NLP & ML",
    top_n=20,
)

# Cell 2 — print the report
print_report(report)

# Cell 3 — save to JSON
import os, json
os.makedirs("outputs", exist_ok=True)
with open("outputs/skill_match_report.json", "w") as f:
    json.dump(report.to_dict(), f, indent=2)
print("Saved.")


════════════════════════════════════════════════════════════
  JOB: Senior Data Scientist – NLP & ML
  MATCH SCORE: 41.7%
════════════════════════════════════════════════════════════

✅  MATCHED SKILLS  (7)
────────────────────────────────────────────────────────────
Skill                         Relevance  Difficulty     Category
────────────────────────────────────────────────────────────
  python                         100.0  Intermediate   Programming
  sql                            100.0  Intermediate   Data Engineering
  machine learning               100.0  Advanced       ML/AI
  aws                             50.0  Intermediate   Cloud
  docker                          50.0  Intermediate   Cloud/DevOps
  statistics                      50.0  Intermediate   Analytics
  hypothesis testing              50.0  Intermediate   Analytics

❌  MISSING SKILLS  (13)  — ordered by learning priority
────────────────────────────────────────────────────────────
#    Skill                  

In [ ]:
### testing on my resume sample
my_resume = '''
ERA AHSAN
BUSINESS ANALYST
era_ahsan@berkeley.edu | linkedin.com/eraahsan | (347) 476-1348 | Berkeley, CA | Willing to Relocate | U.S. Citizen
UC Berkeley Master of Analytics candidate with experience translating complex data into strategic business recommendations.
Proven ability to structure ambiguous problems, lead cross-functional analysis, and deliver actionable insights that improve
operational and financial outcomes. Skilled at combining quantitative modeling, stakeholder engagement, and executive
communication to support high-impact strategic initiatives.
EDUCATION
University of California–College of Engineering, Berkeley
Master of Analytics, May 2026 | Berkeley, CA
Relevant Courses: Optimization Analytics, Analyzing & Designing Databases, Risk Modeling & Simulation, Machine Learning,
Economics of Supply Chains
Fordham University
BA in Interdisciplinary Mathematics and Economics, May 2024 | Bronx, NY
SKILLS
Strategic & Analytical: Data-Driven Decision Support, Analysis (Business Strategy, Root Cause, Market & Risk,), Structured Problem
Solving, Financial Modeling, Statistical Forecasting
Programs & Software: Python (Pandas, NumPy, Matplotlib, Dash), SQL (PostgreSQL), R (Dplyr, Ggplot), Excel (VLOOKUP, Pivot Tables,
Power Query, Data Validation), PowerPoint, Bloomberg Terminal (BMC Certified, Finance Fundamentals, Bloomberg Client Service)
Languages: Bengali, Spanish, Hindi, Urdu
EXPERIENCE
University of California, Berkeley Berkeley, CA
Student Ambassador – Master of Analytics Program January 2026 – Present
- Advised prospective students through 1:1 strategic consultations, translating complex program outcomes into clear value
propositions that improve engagement and applicant conversion
- Analyzed applicant engagement data to identify recruitment pipeline bottlenecks and deliver insights that improved outreach
targeting
- Collaborated with program leadership to refine recruitment strategy by synthesizing stakeholder feedback and applicant trend
analysis
- Managed high-volume communications across multiple platforms while maintaining strict confidentiality and service accuracy
Google Hackathon Berkeley, CA
Business Strategy Consultant December 2025
- Diagnosed $20M revenue leak from enterprise client churn by redefining churn through an ACV lens and segmenting high-value
accounts, enabling a strategic focus on the top 17% of clients driving 76% of lost revenue
- Applied 80/20 analysis and machine learning (Random Forest Classifier) to identify predictive churn drivers–resolution time,
onboarding score, and discounts– validating insights across usage, support, and billing datasets
- Developed “Enterprise Recovery” recommendation including tiered resolution targets, onboarding interventions, and renewal
discounts, projected to recover $154K in revenue per 1% ACV churn reduction
- Presented actionable strategy to senior stakeholders in an executive-ready format, translating complex quantitative and qualitative
analyses into prioritized interventions for high-value enterprise accounts
Tesla Global Sourcing Case Study Competition Berkeley, CA
Team Lead October 2025
- Led cross-functional team in diagnosing supply chain cost drivers and supplier risk exposure, identifying sourcing strategy
projected to reduce procurement costs by 15%
- Structured fragmented operational data into integrated financial and risk models to support evidence-based supplier allocation
decisions
- Conducted sensitivity and variance analysis to evaluate trade-offs between cost efficiency, delivery reliability, and supplier
concentration risk
- Proposed executive-level recommendations translating complex quantitative findings into actionable sourcing strategies

'''

my_jd = ''' 
Full job description
Salary: $120,000 to $135,000 per annum

Job Summary:

As a Business Analyst, you will play a critical role in supporting client marketing and client loyalty organizations focusing on client locations.
This person will dive deep into our abundant marketing, merchandising, and sales data to find valuable client insights, drive decision making, and influence business strategy. The successful candidate has an innate curiosity, strong analytical/dashboarding skills and experience with complex business analysis.
This position reports to the Analytics Manager, and has broad exposure, visibility, and impact throughout the organization.
Key Responsibilities:

Analyze large volumes of client data and use it to inform how we define client-centric strategies to grow our business.
Conduct weekly and monthly reporting to help uncover new insights, opportunities, and drivers of client loyalty.
Synthesize complex data analytics into easily understood concepts by creating data visualizations and dashboards that will be shared at all levels of management.
Develop, publish, and optimize dashboards using Tableau/Excel to share with business partners and leadership.
Perform ad hoc analysis to answer strategic business questions.
'''


report = match_skills(
    job_description=my_jd,
    resume_text=my_resume,
    job_title="Whatever the job is called",
    top_n=20,
)
print_report(report)

end_time = time.time()
print('time to execute code:', end_time - start_time, 'seconds')


════════════════════════════════════════════════════════════
  JOB: Whatever the job is called
  MATCH SCORE: 66.7%
════════════════════════════════════════════════════════════

✅  MATCHED SKILLS  (2)
────────────────────────────────────────────────────────────
Skill                         Relevance  Difficulty     Category
────────────────────────────────────────────────────────────
  excel                          100.0  Beginner       Analytics
  leadership                     100.0  Intermediate   Soft Skills

❌  MISSING SKILLS  (1)  — ordered by learning priority
────────────────────────────────────────────────────────────
#    Skill                         Relevance  Difficulty     Category
────────────────────────────────────────────────────────────
  1   tableau                        100.0  Beginner       Analytics

time to execute code: 7.335422992706299 seconds
